In [0]:
ENV = dbutils.widgets.get("env").strip()
WEBHOOK = dbutils.secrets.get(scope="obs-alerting", key="teams-webhook")
CAT = "mq_gmdf_dev.oil_obs"
DIGEST_FLOOR = 5

if not ENV:
    raise ValueError("Job parameter 'env' is empty. Define it at the job level.")
print(f"env={ENV}  webhook_configured={bool(WEBHOOK)}  digest_floor={DIGEST_FLOOR}")

env=dev  webhook_configured=False  digest_floor=5


In [0]:
spark.sql(f"""
MERGE INTO {CAT}.runtime_observed t
USING (
  WITH env_sanctioned AS (
    SELECT array_distinct(flatten(collect_list(allowed_transports)))     AS ok_transports,
           array_distinct(flatten(collect_list(allowed_model_configs)))  AS ok_model_configs
    FROM {CAT}.runtime_allowlist WHERE environment = '{ENV}'
  ),
  flagged AS (
    SELECT b.capability, b.transport, b.model_config, b.scheduler_run, b.called_at,
           CASE
             WHEN a.capability IS NULL                                         THEN 'unknown_capability'
             WHEN NOT array_contains(a.allowed_transports,     b.transport)     THEN 'transport_not_allowed'
             WHEN NOT array_contains(a.allowed_model_configs,  b.model_config)  THEN 'model_config_not_allowed'
             WHEN NOT array_contains(a.allowed_scheduler_runs, b.scheduler_run) THEN 'scheduler_run_not_allowed'
           END AS violation_type
    FROM {CAT}.v_llm_bronze b
    LEFT JOIN (SELECT * FROM {CAT}.runtime_allowlist WHERE environment = '{ENV}') a
           ON a.capability = b.capability
    WHERE b.called_at >= current_timestamp() - INTERVAL 7 DAYS
      AND (a.capability IS NULL
           OR NOT array_contains(a.allowed_transports,     b.transport)
           OR NOT array_contains(a.allowed_model_configs,  b.model_config)
           OR NOT array_contains(a.allowed_scheduler_runs, b.scheduler_run))
  )
  SELECT
      '{ENV}' AS environment,
      sha2(concat_ws('|', coalesce(f.capability,'<null>'), coalesce(f.transport,'<null>'),
                          coalesce(f.model_config,'<null>'), coalesce(f.scheduler_run,'<null>'),
                          coalesce(f.violation_type,'<null>')), 256) AS violation_signature,
      f.capability, f.transport, f.model_config, f.scheduler_run, f.violation_type,
      CASE
        WHEN size(coalesce(s.ok_transports, array())) = 0                              THEN 'digest'
        WHEN NOT array_contains(s.ok_transports,    coalesce(f.transport,'<null>'))     THEN 'immediate'
        WHEN NOT array_contains(s.ok_model_configs, coalesce(f.model_config,'<null>'))  THEN 'immediate'
        ELSE 'digest'
      END AS violation_tier,
      min(f.called_at) AS first_seen,
      max(f.called_at) AS last_seen,
      count(*)         AS occurrences_7d
  FROM flagged f CROSS JOIN env_sanctioned s
  GROUP BY 1,2,3,4,5,6,7,8
) s
ON t.environment = s.environment AND t.violation_signature = s.violation_signature
WHEN MATCHED THEN UPDATE SET
    t.last_seen      = s.last_seen,
    t.occurrences_7d = s.occurrences_7d,
    t.violation_tier = s.violation_tier
WHEN NOT MATCHED THEN INSERT
    (environment, violation_signature, capability, transport, model_config, scheduler_run,
     violation_type, violation_tier, first_seen, last_seen, occurrences_7d)
  VALUES
    (s.environment, s.violation_signature, s.capability, s.transport, s.model_config,
     s.scheduler_run, s.violation_type, s.violation_tier, s.first_seen, s.last_seen,
     s.occurrences_7d)
""")
print("runtime_observed updated.")

runtime_observed updated.


In [0]:
%sql
-- should be populated now, non-zero
SELECT count(*) AS n FROM mq_gmdf_dev.oil_obs.runtime_observed WHERE environment = 'dev';

-- 1:1 guarantee, same shape check as Step 1
SELECT count(*) AS rows, count(DISTINCT violation_signature) AS signatures
FROM mq_gmdf_dev.oil_obs.runtime_observed WHERE environment = 'dev';

-- eyeball it: should show cortex-claude46 AND spe-claude-sonnet-46 as 'immediate',
-- and the test-bot-claude-v1 renames as 'digest'
SELECT capability, model_config, transport, violation_tier, occurrences_7d,
       first_seen, last_seen, disposition
FROM mq_gmdf_dev.oil_obs.runtime_observed
WHERE environment = 'dev'
ORDER BY violation_tier, occurrences_7d DESC;

-- re-run Cell 2 a second time immediately after. occurrences_7d should UPDATE
-- in place (same signatures), not duplicate. Row count from the query above
-- should be identical before and after.

capability,model_config,transport,violation_tier,occurrences_7d,first_seen,last_seen,disposition
dsa_copilot_step,test-bot-claude-v1,cortex,digest,507,2026-08-23T22:47:26.082Z,2026-08-27T17:11:51.387Z,null
saa_insight,test-bot-claude-v1,cortex,digest,92,2026-08-23T22:46:43.743Z,2026-08-27T16:32:39.261Z,null
dsa_copilot_step,vertex26,cortex,digest,25,2026-08-22T19:08:34.974Z,2026-08-22T20:49:17.838Z,null
dsa_compare,test-bot-claude-v1,cortex,digest,16,2026-08-24T13:35:38.789Z,2026-08-24T17:50:33.003Z,null
dsa_session_summary,test-bot-claude-v1,cortex,digest,13,2026-08-24T14:55:51.681Z,2026-08-27T17:13:23.303Z,null
sev2_insight,test-bot-claude-v1,cortex,digest,8,2026-08-26T14:02:42.957Z,2026-08-26T15:47:03.583Z,null
dsa_session_summary,vertex26,cortex,digest,6,2026-08-22T19:12:58.374Z,2026-08-23T01:37:48.077Z,null
dsa_batch_summary,test-bot-claude-v1,cortex,digest,3,2026-08-23T22:57:45.526Z,2026-08-24T23:32:50.815Z,null
probe,test-bot-claude-v1,cortex,digest,1,2026-08-23T22:47:05.105Z,2026-08-23T22:47:05.105Z,null
dsa_copilot,test-bot-claude-v1,cortex,digest,1,2026-08-27T04:13:01.503Z,2026-08-27T04:13:01.503Z,null


In [ ]:
import json
from datetime import datetime, timezone

rows = spark.sql(f"""
    SELECT ro.capability, ro.transport, ro.model_config, ro.scheduler_run, ro.violation_type,
           ro.occurrences_7d, ro.first_seen, ro.last_seen,
           datediff(current_timestamp(), ro.first_seen) AS age_days,
           coalesce(r.owner, 'unassigned') AS owner
    FROM {CAT}.runtime_observed ro
    LEFT JOIN {CAT}.capability_registry r ON r.capability = ro.capability
    WHERE ro.environment = '{ENV}'
      AND ro.disposition IS NULL
      AND ro.violation_tier = 'digest'
      AND (ro.digest_reported_at IS NULL
           OR ro.digest_reported_at < current_timestamp() - INTERVAL 6 DAYS)
      AND ro.occurrences_7d >= {DIGEST_FLOOR}
    ORDER BY ro.occurrences_7d DESC
""").collect()

below = spark.sql(f"""
    SELECT count(*) AS n FROM {CAT}.runtime_observed
    WHERE environment = '{ENV}' AND disposition IS NULL
      AND violation_tier = 'digest'
      AND occurrences_7d < {DIGEST_FLOOR}
""").collect()[0]["n"]

by_capability = {}
if not rows:
    print("No undispositioned runtime configurations above the floor. Nothing to digest.")
    print(f"({below} configuration(s) exist below the {DIGEST_FLOOR}-call floor, if any.)")
else:
    # Group by capability. A capability can appear multiple times with different
    # model_config/transport/scheduler_run — each is its own row in runtime_observed.
    # Emitting one INSERT per row would create duplicate capability rows in
    # runtime_allowlist, which every detector joins on capability alone — a second
    # matching row makes a legitimate call fail the array_contains check on the OTHER
    # row and get flagged as a violation. Group first; emit one INSERT per capability
    # with every unsanctioned model_config folded into one array.
    for r in rows:
        by_capability.setdefault(r["capability"], []).append(r)

    def sanction_sql_grouped(capability, group):
        transports     = sorted({g["transport"] for g in group})
        model_configs  = sorted({g["model_config"] for g in group})
        scheduler_runs = sorted({g["scheduler_run"] for g in group})
        total_calls    = sum(g["occurrences_7d"] for g in group)
        earliest       = min(g["first_seen"] for g in group)

        transport_arr  = ", ".join(f"'{t}'" for t in transports)
        model_arr      = ", ".join(f"'{m}'" for m in model_configs)
        scheduler_arr  = ", ".join(f"'{s}'" for s in scheduler_runs)
        model_list_str = ", ".join(model_configs)

        detail = "; ".join(
            f"{g['model_config']}={g['occurrences_7d']} calls (first {g['first_seen']:%Y-%m-%d})"
            for g in sorted(group, key=lambda g: -g["occurrences_7d"])
        )

        return (
            f"-- {capability} · {total_calls} calls in 7d across {len(model_configs)} "
            f"model_config(s): {model_list_str}\n"
            f"-- breakdown: {detail}\n"
            f"-- REVIEW EACH model_config before running — stale ones (see 'last_seen' in "
            f"runtime_observed) may belong in 'transient', not the allowlist.\n"
            f"INSERT INTO {CAT}.runtime_allowlist VALUES ('{ENV}', '{capability}', "
            f"array({transport_arr}), array({model_arr}), array({scheduler_arr}));\n"
            f"UPDATE {CAT}.runtime_observed SET disposition='sanctioned', "
            f"dispositioned_by='you@lilly.com', dispositioned_at=current_timestamp() "
            f"WHERE environment='{ENV}' AND capability='{capability}';"
        )

    body = [{
        "type": "Container", "style": "accent", "bleed": True,
        "items": [
            {"type": "TextBlock", "size": "Large", "weight": "Bolder",
             "text": f"Weekly runtime configuration digest — {len(rows)} awaiting review"},
            {"type": "TextBlock", "spacing": "None", "isSubtle": True, "wrap": True,
             "text": f"ISH agent observability · env **{ENV}** · "
                     f"{datetime.now(timezone.utc):%Y-%m-%d} UTC · "
                     f"nothing here has been auto-approved"},
        ],
    }]

    for capability, group in by_capability.items():
        model_configs  = sorted({g["model_config"] for g in group})
        transports     = sorted({g["transport"] for g in group})
        scheduler_runs = sorted({g["scheduler_run"] for g in group})
        total_calls    = sum(g["occurrences_7d"] for g in group)
        earliest       = min(g["first_seen"] for g in group)
        violation_types = sorted({g["violation_type"] for g in group})
        owner          = group[0]["owner"]  # 1:1 per capability; already coalesced to 'unassigned' in SQL

        body.append({"type": "Container", "separator": True, "spacing": "Medium", "items": [
            {"type": "TextBlock", "weight": "Bolder", "wrap": True,
             "text": f"{capability} — {', '.join(violation_types)}"},
            {"type": "FactSet", "facts": [
                {"title": "Owner",             "value": owner},
                {"title": "Transport(s)",     "value": ", ".join(transports)},
                {"title": "Model config(s)",  "value": ", ".join(model_configs)
                 + (" ⚠ multiple — review before sanctioning" if len(model_configs) > 1 else "")},
                {"title": "Scheduler run(s)", "value": ", ".join(scheduler_runs)},
                {"title": "Calls (7d)",        "value": str(total_calls)},
                {"title": "First seen",        "value": f"{earliest:%Y-%m-%d %H:%M}"},
            ]},
            {"type": "TextBlock", "wrap": True, "fontType": "Monospace", "size": "Small",
             "isSubtle": True,
             "text": sanction_sql_grouped(capability, group).replace("\n", "<br>")},
        ]})

    if below:
        body.append({"type": "TextBlock", "separator": True, "wrap": True, "isSubtle": True,
                     "text": f"{below} further configuration(s) below the {DIGEST_FLOOR}-call "
                             f"floor are not listed. Query runtime_observed to see them."})

    card = {"type": "AdaptiveCard", "version": "1.4", "body": body,
            "$schema": "http://adaptivecards.io/schemas/adaptive-card.json"}

    if WEBHOOK:
        import requests
        try:
            resp = requests.post(WEBHOOK, data=json.dumps({"type": "message", "attachments": [
                {"contentType": "application/vnd.microsoft.card.adaptive",
                 "content": card}]}),
                headers={"Content-Type": "application/json"}, timeout=30)
            print(f"Teams webhook status={resp.status_code}")
            if resp.status_code < 400:
                spark.sql(f"""
                    UPDATE {CAT}.runtime_observed
                    SET digest_reported_at = current_timestamp()
                    WHERE environment = '{ENV}' AND disposition IS NULL
                      AND violation_tier = 'digest'
                      AND occurrences_7d >= {DIGEST_FLOOR}
                """)
        except Exception as e:  # noqa: BLE001
            print(f"Teams webhook FAILED: {str(e)[:200]}")
    else:
        print("No webhook configured; card not posted.")
        for capability, group in by_capability.items():
            print(sanction_sql_grouped(capability, group), "\n")

In [0]:
print(len(by_capability), "distinct capabilities in this digest")
for cap, group in by_capability.items():
    if len(group) > 1:
        print(f"  {cap}: {len(group)} model_configs -> {[g['model_config'] for g in group]}")

0 distinct capabilities in this digest


In [0]:
immediate_leak = spark.sql(f"""
    SELECT concat(capability, '/', model_config) AS bad
    FROM {CAT}.runtime_observed
    WHERE environment = '{ENV}' AND violation_tier = 'immediate'
      AND concat(capability, '/', model_config) IN (
        {", ".join(f"'{g['capability']}/{g['model_config']}'" for group in by_capability.values() for g in group)}
      )
""").collect() if by_capability else []
assert not immediate_leak, f"IMMEDIATE-tier config leaked into digest: {immediate_leak}"
print("OK: no immediate-tier configs in digest grouping.")

OK: no immediate-tier configs in digest grouping.
